# Tutorial 02b

Mathematics for Machine Learning

Thomas Manke  
2026-10-26

Our goal is to build a mathematical model that explains **Annual Medical
Charges ($y$, in USD)** as a function of **Body Mass Index ($x$, in
$\text{kg/m}^2$)**: $$y = f(x)$$

To achieve this, the notebook covers several foundational steps in
machine learning:

- **Phase 1: Exploratory Data Analysis** — Understanding the variables
  and inspecting their empirical distributions.
- **Phase 2: Data Transformations** — Evaluating and selecting suitable
  transformations to improve feature and target representations.
- **Phase 3: Defining and Solving a Model (OLS)** — Formulating and
  solving linear regression via Ordinary Least Squares, followed by
  model critique.
- **Phase 4: Parameter Sensitivity and Resampling** — Investigating
  parameter uncertainty, correlation, and prediction spread through
  empirical resampling.

------------------------------------------------------------------------

## Setup

Run the setup cell below to confirm package installations, report their
versions, and load the raw dataset.

- **Data Exploration:** How many observations and how many variables
  does the dataset contain?
- **Variable Documentation:** Report all variables and explain what each
  measurement represents. Consult the documentation on the
  <a href="https://www.openml.org/d/46289" target="_blank"
  rel="noopener">OpenML Dataset 46289</a> for feature definitions.
- *Hint:* For the subsequent modeling tasks, you will focus on
  predicting medical charges from body mass index (BMI). To avoid the
  confounding effect of smoking status, consider focusing on the cohort
  of non-smokers (`smoker == 'no'`).

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

# Global parameters
SUBSAMPLE_RATIO = 0.80  # Default subsample size: 80% of observations

print(f"Python Version: {sys.version.split()[0]}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"Matplotlib Version: {plt.matplotlib.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")

# Fetch Medical Cost Personal Dataset
try:
    from sklearn.datasets import fetch_openml
    raw_data = fetch_openml("medical_cost", version=1, as_frame=True)
    df = raw_data.frame
except Exception:
    print("Unable to fetch dataset from OpenML.")

df.head()

Python Version: 3.11.16
NumPy Version: 2.4.6
Pandas Version: 3.0.5
Matplotlib Version: 3.11.1
Scikit-Learn Version: 1.9.1

------------------------------------------------------------------------

## Phase 1: Exploratory Data Analysis

- **Numerical Summaries:** Compute and compare numerical summaries for
  both variables (e.g., using pandas `describe()`). Interpret the
  output: what do the central tendencies and spread indicate about each
  variable?
- **Visual Summaries:** Plot side-by-side distribution plots
  (histograms) for `bmi` and `charges`. Are these variables normally
  distributed? What characteristics do you notice, and what could be
  done about them?
- **Feature Relationship:** Create a scatter plot of medical charges
  ($y$) versus BMI ($x$). Interpret the pattern: does a straight line
  appear suitable for describing this relationship directly?

------------------------------------------------------------------------

## Phase 2: Data Transformations

In linear modeling, it is often preferable to work with variables that
have approximately symmetric, well-behaved distributions rather than
heavy skewness or multiplicative variance growth.

- **Evaluating Transformations:** Notice that annual medical charges
  cannot be negative and exhibit a long right-hand tail. Consider
  various transformations and plot the distribution before and after
  transformation. Decide which transformation is most useful in this
  case.
- **Transformed Relationship:** Plot BMI ($x$) against the transformed
  charges ($\tilde{y}$). How has the relationship changed compared to
  the raw dollar scatter plot? Does the variance appear more stable?

------------------------------------------------------------------------

## Phase 3: Defining and Solving a Model (OLS)

We set up a simple linear model relating BMI to the transformed charges:
$$\tilde{y}_i \approx \theta_0 + \theta_1 x_i$$

- **Design Matrix:** Construct the design matrix $\mathbf{X}$ (including
  an intercept column) in NumPy. Print its shape.
- **Gram Matrix & Invertibility:** Construct the Gram matrix. Report its
  shape and determinant, test whether it is invertible, and compute its
  inverse.
- **Normal Equations:** Solve the normal equations to compute the
  parameters of the linear regression.
- **Parameter Interpretation:** Print the estimated parameters and
  explain their physical / financial interpretation.
- **Visualizing the Model Fit:** Plot the fitted regression line over
  the observed data points on both the transformed scale and the
  original dollar scale
  ($y = \exp(\mathbf{X}\hat{\boldsymbol{\theta}})$).
- **Model Critique (Residual Analysis):**
  - Collect the residual errors:
    $e_i = \tilde{y}_i - \hat{\tilde{y}}_i$.
  - Investigate their properties: what is their mean? What is their
    standard deviation?
  - Plot a histogram of the residuals. Are the errors approximately
    normally distributed?
  - Plot the residuals ($e_i$) against the predicted values
    ($\hat{\tilde{y}}_i$). Check for signs of heteroscedasticity or
    remaining non-linear patterns.

------------------------------------------------------------------------

## Phase 4: Parameter Sensitivity and Resampling

Even though our normal equation solver is deterministic, the dataset
itself is a sample drawn from an underlying population.

- **Resampling:** Write a loop over 1000 iterations. In each iteration,
  sample 80% of the observations without replacement and compute the
  regression parameters. Alternatively, sample 100% of the observations
  **with** replacement and compute the regression parameters as before.
- **Parameter Spread:** Prepare histograms for both parameters to
  inspect how their estimates vary across resamples.
- **Parameter Space:** Create a 2D scatter plot of parameter 1 against
  parameter 2 (intercept vs. slope) across all resampling iterations.
  What do you observe about their relationship and correlation?
- **Prediction Spread:** Plot the collection of fitted regression curves
  across the dataset in original dollar terms to observe the financial
  prediction uncertainty.